# Chapter 3 · Bernstein-Vazirani Algorithm

## Objectives

1. Understand the Bernstein-Vazirani problem as an extension of Deutsch's algorithm.
2. Implement the linear oracle and the complete algorithm.
3. Verify that the secret string $\mathbf{s}$ is recovered in a single query.

---

## 3B.1 The Problem

Given the function $f(\mathbf{x}) = \mathbf{s} \cdot \mathbf{x} \pmod{2}$ (dot product modulo 2 with a secret string $\mathbf{s} \in \{0,1\}^n$), Bernstein-Vazirani determines $\mathbf{s}$ with a single oracle call, whereas a classical algorithm requires $n$ queries.

The circuit is identical to Deutsch-Jozsa but the oracle encodes the dot product:

$$U_f|\mathbf{x}\rangle|-\rangle = (-1)^{\mathbf{s}\cdot\mathbf{x}}|\mathbf{x}\rangle|-\rangle$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

In [ ]:
def bv_oracle(secret: str) -> QuantumCircuit:
    """Oracle for the Bernstein-Vazirani algorithm.

    The oracle implements f(x) = s·x mod 2, where s is the secret string.
    Applies CNOT from qubit i to the ancilla qubit when s[i] = '1'.

    Parameters
    ----------
    secret : str
        Secret binary string of length n.
    """
    n = len(secret)
    qc = QuantumCircuit(n + 1, name=f'BV_Oracle({secret})')
    for i, bit in enumerate(reversed(secret)):
        if bit == '1':
            qc.cx(i, n)
    return qc


def bernstein_vazirani(secret: str) -> QuantumCircuit:
    """Full circuit for the Bernstein-Vazirani algorithm."""
    n = len(secret)
    qc = QuantumCircuit(n + 1, n)

    # Ancilla in |1⟩
    qc.x(n)
    qc.barrier()

    # Hadamard on all qubits
    qc.h(range(n + 1))
    qc.barrier()

    # Oracle
    oracle = bv_oracle(secret)
    qc.compose(oracle, inplace=True)
    qc.barrier()

    # Hadamard on input qubits
    qc.h(range(n))
    qc.barrier()

    # Measurement
    qc.measure(range(n), range(n))
    return qc


# Test with different secret strings
backend = AerSimulator()
test_secrets = ['10110', '11111', '10000', '01010', '110110']

print('Verification of the Bernstein-Vazirani algorithm:')
print(f'{"Secret":<15} {"Recovered":<15} {"Correct"}')
print('-' * 40)

for secret in test_secrets:
    qc = bernstein_vazirani(secret)
    job = backend.run(qc, shots=1)
    counts = job.result().get_counts()
    # Result is reversed (Qiskit convention)
    recovered = list(counts.keys())[0][::-1]
    correct = (recovered == secret)
    print(f'{secret:<15} {recovered:<15} {"\u2713" if correct else "\u2717"}')

In [ ]:
# Circuit visualization for a short example
secret_demo = '10110'
qc_demo = bernstein_vazirani(secret_demo)
print(f'Bernstein-Vazirani circuit for s = {secret_demo}:')
print(qc_demo.draw('text'))

## 3B.2 Proposed Exercises

1. Formally prove (by following the state evolution) that the state before measurement is exactly $|\mathbf{s}\rangle|{-}\rangle$.

2. How many oracle calls does a classical deterministic algorithm require? What about a probabilistic one that tolerates a 5% error?

3. Generate a random secret string of length 20 and verify that the algorithm recovers it in a single measurement.